In [2]:
from huggingface_hub import snapshot_download

repo_id = "SprintML/tml26_task2"
local_dir = "/kaggle/working/"

snapshot_download(
    repo_id=repo_id,
    repo_type="model",
    local_dir=local_dir,
    allow_patterns=[
        "target_model/",
        "suspect_models/",
    ],
    local_dir_use_symlinks=False,
    resume_download=True,
)
print("Code files downloaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:186: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 362 files:   0%|          | 0/362 [00:00<?, ?it/s]

Code files downloaded.


In [ ]:
def load_model(path, device="cpu"):
    state_dict = load_file(path, device="cpu")  
    model = make_model()
    model.load_state_dict(state_dict, strict=True)
    model.eval()
    return model.to(device)  

In [ ]:
def load_model(path, device="cpu"):
    state_dict = load_file(path, device="cpu")
    model = make_model()
    model.load_state_dict(state_dict, strict=True)
    model.eval()
    return model.to(device)

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torchvision.models import resnet18
from safetensors.torch import load_file
import pandas as pd
import numpy as np
from tqdm import tqdm


TARGET_CKPT  = "/kaggle/working/target_model/weights.safetensors"      
SUSPECT_DIR  = "/kaggle/working/suspect_models"     
                                                   
DATA_ROOT    = "./data"                            
OUTPUT_CSV   = "submission.csv"
N_SAMPLES    = 500                              
BATCH_SIZE   = 64



def make_model():
    model = resnet18(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, 100)
    return model



def load_model(path, device="cpu"):
    device_str = str(device)
    if device_str == "cuda":
        device_str = "cuda:0"
    state_dict = load_file(path, device=device_str)
    model = make_model()
    model.load_state_dict(state_dict, strict=True)
    model.eval()
    return model.to(device)



def weight_similarity(target, suspect):
    sims = []
    t_params = dict(target.named_parameters())
    s_params = dict(suspect.named_parameters())
    for name, tp in t_params.items():
        if name not in s_params:
            continue
        sp = s_params[name]
        if tp.shape != sp.shape:
            continue
        t_flat = tp.detach().float().cpu().flatten()
        s_flat = sp.detach().float().cpu().flatten()
        cos = F.cosine_similarity(t_flat.unsqueeze(0), s_flat.unsqueeze(0)).item()
        sims.append((cos + 1.0) / 2.0)  
    return float(np.mean(sims)) if sims else 0.0



@torch.no_grad()
def collect_logits(model, loader, device):
    all_probs = []
    for x, _ in loader:
        x = x.to(device)
        probs = F.softmax(model(x), dim=-1)
        all_probs.append(probs.cpu())
    return torch.cat(all_probs, dim=0) 

def prediction_similarity(target_probs, suspect_probs):
    cos = F.cosine_similarity(target_probs, suspect_probs, dim=1)  
    return cos.mean().item()



@torch.no_grad()
def collect_features(model, loader, device):
    features = []
    def hook_fn(module, input, output):
        features.append(output.cpu())
    h = model.avgpool.register_forward_hook(hook_fn)
    for x, _ in loader:
        model(x.to(device))
    h.remove()
    return torch.cat(features, dim=0).squeeze(-1).squeeze(-1)  

def cka(X, Y):
    X = X.float() - X.float().mean(0)
    Y = Y.float() - Y.float().mean(0)
    Kx = X @ X.T
    Ky = Y @ Y.T
    num   = (Kx * Ky).sum()
    denom = ((Kx * Kx).sum() * (Ky * Ky).sum()).sqrt()
    return (num / denom).item() if denom > 1e-10 else 0.0

def activation_similarity(target_feats, suspect_feats):
    return cka(target_feats, suspect_feats)



def find_checkpoints(suspect_dir):
    """
    Supports two layouts:
      HuggingFace style: <suspect_dir>/<id>/model.safetensors
      Flat style:        <suspect_dir>/<id>.safetensors
    Returns dict {id (int): path (str)}
    """
    from pathlib import Path
    suspect_dir = Path(suspect_dir)
    ckpts = {}

  
    for subdir in suspect_dir.iterdir():
        if subdir.is_dir():
            for fname in ["model.safetensors", "pytorch_model.bin"]:
                ckpt = subdir / fname
                if ckpt.exists():
                    try:
                        ckpts[int(subdir.name)] = str(ckpt)
                    except ValueError:
                        pass

    if not ckpts:
        for f in suspect_dir.glob("*.safetensors"):
            stem = f.stem.replace("suspect_", "") 
            try:
                ckpts[int(stem)] = str(f)
            except ValueError:
                pass

    return ckpts



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408),
                         (0.2675, 0.2565, 0.2761)),
])
dataset = datasets.CIFAR100(root=DATA_ROOT, train=False, download=True, transform=transform)
indices = torch.randperm(len(dataset))[:N_SAMPLES].tolist()
subset  = torch.utils.data.Subset(dataset, indices)
loader  = torch.utils.data.DataLoader(subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("Loading target model...")
target = load_model(TARGET_CKPT, device)
target_probs = collect_logits(target, loader, device)
target_feats = collect_features(target, loader, device)

ckpts = find_checkpoints(SUSPECT_DIR)
print(f"Found {len(ckpts)} suspect checkpoints.")

subset_ids         = list(range(360))
confidence_scores  = []

for idx in tqdm(subset_ids, desc="Scoring"):
    if idx not in ckpts:
        confidence_scores.append(0.5) 
        continue
    try:
        suspect = load_model(ckpts[idx], device)

        w_sim = weight_similarity(target, suspect)
        p_sim = prediction_similarity(target_probs, collect_logits(suspect, loader, device))
        a_sim = activation_similarity(target_feats, collect_features(suspect, loader, device))

       
        score = 0.4 * w_sim + 0.4 * p_sim + 0.2 * a_sim
        confidence_scores.append(score)

        del suspect
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    except Exception as e:
        print(f"  [!] Error on model {idx}: {e}")
        confidence_scores.append(0.0)

scores = np.array(confidence_scores)
lo, hi = scores.min(), scores.max()
if hi - lo > 1e-12:
    scores = (scores - lo) / (hi - lo)

submission_df = pd.DataFrame({
    "id":    subset_ids,
    "score": scores.tolist(),
})
submission_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(submission_df)} rows to {OUTPUT_CSV}")

assert submission_df["id"].min()    == 0,   "id min != 0"
assert submission_df["id"].max()    == 359, "id max != 359"
assert submission_df["id"].nunique() == 360, "duplicate ids"
assert submission_df["score"].notna().all(), "NaN scores"
assert np.isfinite(submission_df["score"].to_numpy()).all(), "Inf scores"
print("Validation passed ✓")

print("\nTop 20 highest-confidence stolen models:")
print(submission_df.nlargest(20, "score").to_string(index=False))

Device: cpu
Loading target model...
Found 360 suspect checkpoints.



Scoring: 100%|██████████| 360/360 [1:25:50<00:00, 14.31s/it]

Saved 360 rows to submission.csv
Validation passed ✓

Top 20 highest-confidence stolen models:
 id    score
 71 1.000000
124 1.000000
138 1.000000
145 1.000000
358 1.000000
224 0.998232
 78 0.996706
313 0.996701
168 0.996657
259 0.996479
341 0.996466
 45 0.995884
104 0.995825
195 0.995732
 14 0.995579
218 0.995143
148 0.994918
 54 0.993128
  8 0.992912
198 0.992652
